# Chapter 17 — Hard Problems

Twenty-six harder problems. Each has a paraphrased prompt, an optimized solution, and tests. Most are checked against a brute-force reference.

In [1]:
from __future__ import annotations

import bisect
import heapq
import itertools
import math
import random
import string
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from functools import cache

## 17.1 Add Without Plus
Add two numbers without `+` or any arithmetic operator. XOR gives the sum without carries, and `(a & b) << 1` gives the carries. Repeat until no carry is left. Python ints are unbounded, so we emulate 32-bit wrap-around; otherwise negative numbers would loop forever.

In [2]:
MASK = 0xFFFFFFFF


def add(a: int, b: int) -> int:
    a, b = a & MASK, b & MASK
    while b:
        a, b = (a ^ b) & MASK, ((a & b) << 1) & MASK
    return a - (1 << 32) if a >> 31 else a


for _ in range(1000):
    x, y = random.randint(-10**9, 10**9), random.randint(-10**9, 10**9)
    assert add(x, y) == x + y
print("17.1 ok")

17.1 ok


## 17.2 Shuffle
Shuffle a deck so all 52! orderings are equally likely, given a perfect random number generator. **Fisher–Yates**: for i from the end, swap `a[i]` with a random `a[j]`, j ≤ i. (This is what `random.shuffle` does.)

In [3]:
def shuffle(a: list) -> list:
    for i in range(len(a) - 1, 0, -1):
        j = random.randint(0, i)
        a[i], a[j] = a[j], a[i]
    return a


counts = Counter(tuple(shuffle([1, 2, 3])) for _ in range(60_000))
assert len(counts) == 6 and all(9_000 < c < 11_000 for c in counts.values())
print("17.2 ok", dict(counts))

17.2 ok {(1, 3, 2): 9920, (2, 3, 1): 10015, (3, 1, 2): 10116, (3, 2, 1): 10019, (1, 2, 3): 10080, (2, 1, 3): 9850}


## 17.3 Random Set
Pick m of n elements at random, with every subset equally likely. **Reservoir sampling**: fill the reservoir with the first m, then for element i ≥ m, replace a random slot with probability m/(i+1). It works on streams of unknown length.

In [4]:
def random_subset(xs, m: int) -> list:
    reservoir = []
    for i, x in enumerate(xs):
        if i < m:
            reservoir.append(x)
        else:
            j = random.randint(0, i)
            if j < m:
                reservoir[j] = x
    return reservoir


hits = Counter(x for _ in range(30_000) for x in random_subset(range(10), 3))
assert all(8_000 < c < 10_000 for c in hits.values())  # each element appears ~ 3/10 of the time
print("17.3 ok")

17.3 ok


## 17.4 Missing Number
An array holds every integer 0..n except one. You can't read `A[i]` whole, only its j-th bit, in constant time. Find the missing number in `O(n)`. Look at the least significant bits. If fewer numbers end in 0 than should, the missing number ends in 0. Keep only the numbers matching that bit, then recurse on the next bit. The total work is n + n/2 + n/4 + … = `O(n)`.

In [5]:
class BitArray:
    def __init__(self, xs: list[int]) -> None:
        self._xs = xs
        self.reads = 0

    def fetch(self, i: int, j: int) -> int:
        self.reads += 1
        return (self._xs[i] >> j) & 1

    def __len__(self) -> int:
        return len(self._xs)


def find_missing(arr: BitArray) -> int:
    def go(indices: list[int], column: int) -> int:
        if not indices:
            return 0
        zeros = [i for i in indices if arr.fetch(i, column) == 0]
        ones = [i for i in indices if arr.fetch(i, column) == 1]  # (re-reads only for clarity)
        if len(zeros) <= len(ones):
            return (go(zeros, column + 1) << 1) | 0
        return (go(ones, column + 1) << 1) | 1

    return go(list(range(len(arr))), 0)


for n in [1, 2, 7, 100, 1000]:
    for missing in {0, n // 2, n}:
        nums = [x for x in range(n + 1) if x != missing]
        random.shuffle(nums)
        ba = BitArray(nums)
        assert find_missing(ba) == missing
        assert ba.reads <= 4 * (n + 1) + 10  # linear
print("17.4 ok")

17.4 ok


## 17.5 Letters and Numbers
Given an array of letters and digits, find the longest subarray with equally many letters and digits. Track the running `letters - digits` difference. Two positions with the same difference bound a balanced subarray, so store the first index where each difference appears: `O(n)`.

In [6]:
def longest_balanced(a: list[str]) -> list[str]:
    first_seen = {0: -1}
    diff, best = 0, (0, -1)
    for i, ch in enumerate(a):
        diff += 1 if ch.isalpha() else -1
        if diff in first_seen:
            start = first_seen[diff]
            if i - start > best[1] - best[0]:
                best = (start, i)
        else:
            first_seen[diff] = i
    return a[best[0] + 1:best[1] + 1]


def longest_balanced_brute(a):
    best = 0
    for i in range(len(a)):
        for j in range(i, len(a)):
            sub = a[i:j + 1]
            if sum(c.isalpha() for c in sub) * 2 == len(sub):
                best = max(best, len(sub))
    return best


arr = list("aa1a1111aaa1a")
assert len(longest_balanced(arr)) == longest_balanced_brute(arr)
for _ in range(200):
    r = [random.choice("a1") for _ in range(random.randint(0, 14))]
    assert len(longest_balanced(r)) == longest_balanced_brute(r)
print("17.5 ok")

17.5 ok


## 17.6 Count of 2s
Count how many times the digit 2 appears across all numbers from 0 to n. Work digit by digit. For each position, the count depends on whether the current digit is < 2, = 2, or > 2: `O(log n)`.

In [7]:
def count_2s_in_range(n: int) -> int:
    count, power = 0, 1
    while power <= n:
        higher, digit, lower = n // (power * 10), (n // power) % 10, n % power
        if digit < 2:
            count += higher * power
        elif digit == 2:
            count += higher * power + lower + 1
        else:
            count += (higher + 1) * power
        power *= 10
    return count


assert count_2s_in_range(25) == 9  # 2, 12, 20-25 (22 has two)
assert all(count_2s_in_range(n) == sum(str(i).count("2") for i in range(n + 1)) for n in range(0, 3000, 7))
print("17.6 ok")

17.6 ok


## 17.7 Baby Names
You get name frequencies plus pairs of synonymous names. Synonymy is transitive. Report the total frequency for each group under one representative name. **Union-Find** merges the groups in nearly `O(1)` per pair.

In [8]:
class UnionFind:
    def __init__(self) -> None:
        self.parent: dict[str, str] = {}

    def find(self, x: str) -> str:
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path halving
            x = self.parent[x]
        return x

    def union(self, a: str, b: str) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            if rb < ra:
                ra, rb = rb, ra
            self.parent[rb] = ra  # alphabetically-first name becomes the representative


def baby_names(freqs: dict[str, int], synonyms: list[tuple[str, str]]) -> dict[str, int]:
    uf = UnionFind()
    for a, b in synonyms:
        uf.union(a, b)
    totals: Counter[str] = Counter()
    for name, f in freqs.items():
        totals[uf.find(name)] += f
    return dict(totals)


freqs = {"Jon": 10, "John": 15, "Johnny": 25, "Chris": 13, "Kris": 4, "Christopher": 19}
syn = [("Jon", "John"), ("John", "Johnny"), ("Chris", "Kris"), ("Chris", "Christopher")]
assert baby_names(freqs, syn) == {"John": 50, "Chris": 36}
print("17.7 ok")

17.7 ok


## 17.8 Circus Tower
In a human tower, each person must be both shorter and lighter than the person below. Given everyone's (height, weight), find the tallest tower. Sort by height ascending, and for equal heights by weight **descending** so two people with the same height can't stack. Then find the longest strictly increasing subsequence of weights with patience sorting: `O(n log n)`.

In [9]:
def circus_tower(people: list[tuple[int, int]]) -> int:
    people = sorted(people, key=lambda p: (p[0], -p[1]))
    tails: list[int] = []
    for _, w in people:
        i = bisect.bisect_left(tails, w)
        if i == len(tails):
            tails.append(w)
        else:
            tails[i] = w
    return len(tails)


def circus_brute(people):
    best = 0
    for r in range(1, len(people) + 1):
        for combo in itertools.combinations(sorted(people), r):
            if all(a[0] < b[0] and a[1] < b[1] for a, b in zip(combo, combo[1:])):
                best = r
    return best


assert circus_tower([(65, 100), (70, 150), (56, 90), (75, 190), (60, 95), (68, 110)]) == 6
for _ in range(100):
    ppl = [(random.randint(1, 6), random.randint(1, 6)) for _ in range(7)]
    assert circus_tower(ppl) == circus_brute(ppl)
print("17.8 ok")

17.8 ok


## 17.9 Kth Multiple
Find the k-th number whose only prime factors are 3, 5, and 7. The sequence starts at 1. Keep three queues (×3, ×5, ×7). The smallest head is the next number, and each queue only receives multiples that no other queue will also produce: `O(k)`.

In [10]:
def kth_multiple(k: int) -> int:
    if k <= 0:
        raise ValueError
    q3, q5, q7 = deque([1]), deque(), deque()
    val = 0
    for _ in range(k):
        val = min(q3[0] if q3 else math.inf, q5[0] if q5 else math.inf, q7[0] if q7 else math.inf)
        if q3 and val == q3[0]:
            q3.popleft(); q3.append(3 * val); q5.append(5 * val)
        elif q5 and val == q5[0]:
            q5.popleft(); q5.append(5 * val)
        else:
            q7.popleft()
        q7.append(7 * val)
    return int(val)


def kth_brute(k):
    nums = sorted(3**a * 5**b * 7**c for a in range(20) for b in range(15) for c in range(12))
    return nums[k - 1]


assert [kth_multiple(i) for i in range(1, 11)] == [1, 3, 5, 7, 9, 15, 21, 25, 27, 35]
assert all(kth_multiple(k) == kth_brute(k) for k in range(1, 300))
print("17.9 ok")

17.9 ok

## 17.10 Majority Element
Find the value that makes up more than half of an array, in `O(n)` time and `O(1)` space. **Boyer–Moore voting** finds a candidate, then a second pass confirms it.

In [11]:
def majority_element(a: list[int]) -> int | None:
    candidate, count = None, 0
    for x in a:
        if count == 0:
            candidate = x
        count += 1 if x == candidate else -1
    return candidate if a.count(candidate) > len(a) // 2 else None


assert majority_element([1, 2, 5, 9, 5, 9, 5, 5, 5]) == 5
assert majority_element([3, 1, 7, 1, 3, 7, 3, 7, 1, 7, 7]) is None
print("17.10 ok")

17.10 ok


## 17.11 Word Distance
Given a large text file, find the shortest distance (counted in words) between two given words. If this will be asked many times, precompute each word's list of positions. Then merge the two sorted lists with two pointers: `O(p + q)` per query.

In [12]:
class WordDistance:
    def __init__(self, words: list[str]) -> None:
        self.pos: dict[str, list[int]] = defaultdict(list)
        for i, w in enumerate(words):
            self.pos[w].append(i)

    def shortest(self, a: str, b: str) -> int:
        pa, pb = self.pos[a], self.pos[b]
        i = j = 0
        best = math.inf
        while i < len(pa) and j < len(pb):
            best = min(best, abs(pa[i] - pb[j]))
            if pa[i] < pb[j]:
                i += 1
            else:
                j += 1
        return int(best)


text = "the quick brown fox jumps over the lazy dog then the fox sleeps".split()
wd = WordDistance(text)
assert wd.shortest("fox", "the") == 1 and wd.shortest("quick", "dog") == 7
print("17.11 ok")

17.11 ok


## 17.12 BiNode
A `BiNode` has two pointers, `node1` and `node2`. They can serve as a binary tree's left and right, or as a doubly linked list's prev and next. Convert a BST in place into a sorted doubly linked list. Recurse, and return the (head, tail) of each converted subtree: `O(n)`.

In [13]:
@dataclass(eq=False)
class BiNode:
    val: int
    node1: BiNode | None = None
    node2: BiNode | None = None


def bst_to_dll(root: BiNode | None) -> tuple[BiNode | None, BiNode | None]:
    if root is None:
        return None, None
    lhead, ltail = bst_to_dll(root.node1)
    rhead, rtail = bst_to_dll(root.node2)
    root.node1, root.node2 = ltail, rhead
    if ltail: ltail.node2 = root
    if rhead: rhead.node1 = root
    return lhead or root, rtail or root


def insert(root, v):
    if root is None:
        return BiNode(v)
    if v < root.val: root.node1 = insert(root.node1, v)
    else: root.node2 = insert(root.node2, v)
    return root


vals = random.sample(range(100), 15)
root = None
for v in vals:
    root = insert(root, v)
head, tail = bst_to_dll(root)
forward, n = [], head
while n: forward.append(n.val); n = n.node2
backward, n = [], tail
while n: backward.append(n.val); n = n.node1
assert forward == sorted(vals) and backward == sorted(vals, reverse=True)
print("17.12 ok")

17.12 ok


## 17.13 Re-Space
Spaces and capitalization were stripped from a document. Using a dictionary, re-insert the spaces so that as few characters as possible are left unrecognized. DP over positions. A trie stops the inner loop as soon as no word can match: `O(n²)` worst case.

In [14]:
def respace(dictionary: set[str], s: str) -> tuple[int, str]:
    trie: dict = {}
    for w in dictionary:
        node = trie
        for c in w:
            node = node.setdefault(c, {})
        node["$"] = True

    n = len(s)
    # best[i] = (invalid chars, tokens) for s[i:]; a token is [text, recognized?]
    best: list[tuple[int, list[list]]] = [(0, [])] * (n + 1)
    for i in range(n - 1, -1, -1):
        # option 1: s[i] is unrecognized (merge with a following unrecognized run)
        inv, toks = best[i + 1]
        if toks and not toks[0][1]:
            choice = (inv + 1, [[s[i] + toks[0][0], False]] + toks[1:])
        else:
            choice = (inv + 1, [[s[i], False]] + toks)
        # option 2: a dictionary word starts at i
        node = trie
        for j in range(i, n):
            node = node.get(s[j])
            if node is None:
                break
            if "$" in node and best[j + 1][0] < choice[0]:
                choice = (best[j + 1][0], [[s[i:j + 1], True]] + best[j + 1][1])
        best[i] = choice
    invalid, tokens = best[0]
    return invalid, " ".join(t if ok else t.upper() for t, ok in tokens)  # unknown runs upper-cased


words = {"looked", "just", "like", "her", "brother", "look"}
invalid, parsed = respace(words, "jesslookedjustliketimherbrother")
assert invalid == 7 and parsed == "JESS looked just like TIM her brother"
print(invalid, "|", parsed)

7 | JESS looked just like TIM her brother


## 17.14 Smallest K
Find the k smallest numbers in an array.
- A max-heap of size k: `O(n log k)`. Or just `heapq.nsmallest`.
- **Quickselect**: `O(n)` on average.

In [15]:
def smallest_k_heap(a: list[int], k: int) -> list[int]:
    if k <= 0:
        return []
    heap = [-x for x in a[:k]]
    heapq.heapify(heap)
    for x in a[k:]:
        if x < -heap[0]:
            heapq.heapreplace(heap, -x)
    return sorted(-x for x in heap)


def smallest_k_quickselect(a: list[int], k: int) -> list[int]:
    a = a[:]
    lo, hi = 0, len(a) - 1
    while lo < hi:
        pivot = a[random.randint(lo, hi)]
        # 3-way partition (handles duplicates)
        lt, i, gt = lo, lo, hi
        while i <= gt:
            if a[i] < pivot: a[lt], a[i] = a[i], a[lt]; lt += 1; i += 1
            elif a[i] > pivot: a[i], a[gt] = a[gt], a[i]; gt -= 1
            else: i += 1
        if k - 1 < lt: hi = lt - 1
        elif k - 1 > gt: lo = gt + 1
        else: break
    return sorted(a[:k])


for _ in range(200):
    xs = [random.randint(0, 50) for _ in range(random.randint(1, 40))]
    k = random.randint(0, len(xs))
    assert smallest_k_heap(xs, k) == smallest_k_quickselect(xs, k) == sorted(xs)[:k] == heapq.nsmallest(k, xs)
print("17.14 ok")

17.14 ok


## 17.15 Longest Word
Given a word list, find the longest word that can be built by concatenating other words from the list. Check candidates from longest to shortest, with a memoized recursive split.

In [16]:
def longest_compound(words: list[str]) -> str | None:
    word_set = set(words)

    @cache
    def can_build(w: str, is_original: bool) -> bool:
        if w in word_set and not is_original:
            return True
        return any(w[:i] in word_set and can_build(w[i:], False) for i in range(1, len(w)))

    for w in sorted(words, key=lambda w: (-len(w), w)):
        if can_build(w, True):
            return w
    return None


assert longest_compound(["cat", "banana", "dog", "nana", "walk", "walker", "dogwalker"]) == "dogwalker"
assert longest_compound(["a", "b"]) is None
print("17.15 ok")

17.15 ok


## 17.16 The Masseuse
A masseuse gets a list of back-to-back appointment requests (their durations). She can't take two adjacent appointments. Maximize the total booked minutes. This is the classic "house robber" DP, done in `O(n)` time and `O(1)` space.

In [17]:
def masseuse(durations: list[int]) -> int:
    take, skip = 0, 0
    for d in durations:
        take, skip = skip + d, max(take, skip)
    return max(take, skip)


def masseuse_brute(d):
    best = 0
    for mask in range(1 << len(d)):
        if mask & (mask >> 1) == 0:
            best = max(best, sum(d[i] for i in range(len(d)) if mask >> i & 1))
    return best


assert masseuse([30, 15, 60, 75, 45, 15, 15, 45]) == 180
for _ in range(200):
    d = [random.randint(1, 50) for _ in range(random.randint(0, 12))]
    assert masseuse(d) == masseuse_brute(d)
print("17.16 ok")

17.16 ok


## 17.17 Multi Search
Given a string `b` and a list of short strings `T`, find every position in `b` where each string in T occurs. Build a trie of T, then walk it from each start position in `b`: `O(b·k)`, where k is the longest word in T. (For huge inputs, Aho–Corasick reduces this to `O(b + matches)`.)

In [18]:
def multi_search(big: str, smalls: list[str]) -> dict[str, list[int]]:
    trie: dict = {}
    for s in smalls:
        node = trie
        for c in s:
            node = node.setdefault(c, {})
        node["$"] = s
    hits: dict[str, list[int]] = {s: [] for s in smalls}
    for i in range(len(big)):
        node = trie
        for j in range(i, len(big)):
            node = node.get(big[j])
            if node is None:
                break
            if "$" in node:
                hits[node["$"]].append(i)
    return hits


res = multi_search("mississippi", ["is", "ppi", "hi", "sis", "i", "ssippi"])
assert res == {"is": [1, 4], "ppi": [8], "hi": [], "sis": [3], "i": [1, 4, 7, 10], "ssippi": [5]}
print("17.17 ok")

17.17 ok


## 17.18 Shortest Supersequence
Given a short array of distinct values and a longer array, find the shortest subarray of the long one that contains every value in the short one. Sliding window with counts: `O(B)`.

In [19]:
def shortest_supersequence(small: list[int], big: list[int]) -> tuple[int, int] | None:
    need = set(small)
    have: Counter[int] = Counter()
    formed, left = 0, 0
    best: tuple[int, int] | None = None
    for right, x in enumerate(big):
        if x in need:
            have[x] += 1
            if have[x] == 1:
                formed += 1
        while formed == len(need):
            if best is None or right - left < best[1] - best[0]:
                best = (left, right)
            y = big[left]
            if y in need:
                have[y] -= 1
                if have[y] == 0:
                    formed -= 1
            left += 1
    return best


big = [7, 5, 9, 0, 2, 1, 3, 5, 7, 9, 1, 1, 5, 8, 8, 9, 7]
assert shortest_supersequence([1, 5, 9], big) == (7, 10)
assert shortest_supersequence([4], big) is None
print("17.18 ok")

17.18 ok


## 17.19 Missing Two
An array contains every number from 1 to N exactly once except one. Find the missing number in `O(N)` time and `O(1)` space. Follow-up: two numbers are missing. Using the sum and the sum of squares gives two equations in two unknowns. Alternatively, split the range at the midpoint `(x + y) / 2`, since exactly one of the missing numbers falls on each side.

In [20]:
def missing_one(a: list[int]) -> int:
    n = len(a) + 1
    return n * (n + 1) // 2 - sum(a)


def missing_two(a: list[int]) -> tuple[int, int]:
    n = len(a) + 2
    s = n * (n + 1) // 2 - sum(a)                 # x + y
    limit = s // 2                                # x <= limit < y
    x = limit * (limit + 1) // 2 - sum(v for v in a if v <= limit)
    return x, s - x


for n in range(3, 60):
    a, b = sorted(random.sample(range(1, n + 1), 2))
    arr = [v for v in range(1, n + 1) if v not in (a, b)]
    random.shuffle(arr)
    assert missing_two(arr) == (a, b)
    assert missing_one([v for v in range(1, n + 1) if v != a]) == a
print("17.19 ok")

17.19 ok


## 17.20 Continuous Median
Numbers arrive one at a time; keep the median available as they come in. Use two heaps: a max-heap for the lower half and a min-heap for the upper half, balanced so their sizes differ by at most 1. `O(log n)` per insert, `O(1)` to read the median.

In [21]:
class RunningMedian:
    def __init__(self) -> None:
        self.low: list[int] = []   # max-heap via negation
        self.high: list[int] = []  # min-heap

    def add(self, x: int) -> None:
        if not self.low or x <= -self.low[0]:
            heapq.heappush(self.low, -x)
        else:
            heapq.heappush(self.high, x)
        if len(self.low) > len(self.high) + 1:
            heapq.heappush(self.high, -heapq.heappop(self.low))
        elif len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self) -> float:
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2


import statistics
rm, seen = RunningMedian(), []
for _ in range(500):
    x = random.randint(-100, 100)
    rm.add(x); seen.append(x)
    assert rm.median() == statistics.median(seen)
print("17.20 ok")

17.20 ok


## 17.21 Volume of Histogram
A histogram's bars all have width 1. If water were poured over it, how much would it hold? Water above each bar = `min(max_left, max_right) - height`. Two pointers track both maxima: `O(n)` time, `O(1)` space.

In [22]:
def histogram_volume(h: list[int]) -> int:
    left, right = 0, len(h) - 1
    left_max = right_max = water = 0
    while left < right:
        if h[left] < h[right]:
            left_max = max(left_max, h[left])
            water += left_max - h[left]
            left += 1
        else:
            right_max = max(right_max, h[right])
            water += right_max - h[right]
            right -= 1
    return water


def histogram_brute(h):
    return sum(max(0, min(max(h[:i + 1]), max(h[i:])) - h[i]) for i in range(len(h)))


assert histogram_volume([0, 0, 4, 0, 0, 6, 0, 0, 3, 0, 5, 0, 1, 0, 0, 0]) == 26
for _ in range(300):
    hh = [random.randint(0, 9) for _ in range(random.randint(0, 15))]
    assert histogram_volume(hh) == histogram_brute(hh)
print("17.21 ok")

17.21 ok


## 17.22 Word Transformer
Given two words of the same length, transform one into the other by changing one letter at a time, so that every intermediate step is also a dictionary word. BFS gives the shortest path. Bucket words by wildcard patterns (`_amp`, `d_mp`, …) so finding neighbours is fast.

In [23]:
def word_transformer(start: str, end: str, dictionary: set[str]) -> list[str] | None:
    buckets: dict[str, list[str]] = defaultdict(list)
    for w in dictionary | {start}:
        for i in range(len(w)):
            buckets[w[:i] + "_" + w[i + 1:]].append(w)
    parent: dict[str, str | None] = {start: None}
    q = deque([start])
    while q:
        w = q.popleft()
        if w == end:
            path = []
            node: str | None = w
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1]
        for i in range(len(w)):
            for nb in buckets[w[:i] + "_" + w[i + 1:]]:
                if nb not in parent:
                    parent[nb] = w
                    q.append(nb)
    return None


D = {"damp", "dame", "lame", "lamp", "limp", "lime", "like", "lake", "dike"}
path = word_transformer("damp", "like", D)
assert path[0] == "damp" and path[-1] == "like" and len(path) == 5
assert all(sum(a != b for a, b in zip(x, y)) == 1 for x, y in zip(path, path[1:]))
assert word_transformer("damp", "zzzz", D) is None
print(" -> ".join(path))

damp -> lamp -> limp -> lime -> like


## 17.23 Max Black Square
In a square matrix of black and white pixels, find the largest subsquare whose four **borders** are all black. Precompute, for each cell, how many consecutive black cells run to its right and downward. Then each candidate square is checked in `O(1)`, for `O(N³)` total.

In [24]:
def max_black_square(m: list[list[int]]) -> tuple[int, int, int] | None:
    """1 = black. Returns (row, col, size) of the largest square."""
    n = len(m)
    right = [[0] * (n + 1) for _ in range(n + 1)]
    down = [[0] * (n + 1) for _ in range(n + 1)]
    for r in range(n - 1, -1, -1):
        for c in range(n - 1, -1, -1):
            if m[r][c]:
                right[r][c] = right[r][c + 1] + 1
                down[r][c] = down[r + 1][c] + 1
    for size in range(n, 0, -1):
        for r in range(n - size + 1):
            for c in range(n - size + 1):
                if (right[r][c] >= size and down[r][c] >= size
                        and right[r + size - 1][c] >= size and down[r][c + size - 1] >= size):
                    return r, c, size
    return None


M = [[1, 0, 1],
     [0, 1, 1],
     [0, 1, 1]]
assert max_black_square(M) == (1, 1, 2)
M2 = [[1, 1, 1, 1], [1, 0, 0, 1], [1, 0, 0, 1], [1, 1, 1, 1]]
assert max_black_square(M2) == (0, 0, 4)
print("17.23 ok")

17.23 ok


## 17.24 Max Submatrix
Given an N×M matrix of positive and negative ints, find the submatrix with the largest sum. Fix a top row and a bottom row, collapse the columns between them into sums, and run Kadane's algorithm on that: `O(R²·C)`.

In [25]:
def max_submatrix(m: list[list[int]]) -> tuple[int, tuple[int, int, int, int]]:
    rows, cols = len(m), len(m[0])
    best = (-math.inf, (0, 0, 0, 0))
    for top in range(rows):
        col_sums = [0] * cols
        for bottom in range(top, rows):
            for c in range(cols):
                col_sums[c] += m[bottom][c]
            cur, start = 0, 0
            for c in range(cols):
                if cur <= 0:
                    cur, start = col_sums[c], c
                else:
                    cur += col_sums[c]
                if cur > best[0]:
                    best = (cur, (top, start, bottom, c))
    return best  # type: ignore[return-value]


def max_submatrix_brute(m):
    R, C = len(m), len(m[0])
    return max(sum(m[r][c] for r in range(r1, r2 + 1) for c in range(c1, c2 + 1))
               for r1 in range(R) for r2 in range(r1, R) for c1 in range(C) for c2 in range(c1, C))


for _ in range(100):
    mat = [[random.randint(-9, 9) for _ in range(4)] for _ in range(4)]
    assert max_submatrix(mat)[0] == max_submatrix_brute(mat)
print("17.24 ok")

17.24 ok


## 17.25 Word Rectangle
Given a word list, build the largest rectangle of letters in which every row reads left to right as a word and every column reads top to bottom as a word. The words needn't come in any order, and rows must all share a length, as must columns. Try areas from largest to smallest. For each (height, width), place rows by backtracking, and prune with a trie of the words of length `height` so every partial column stays a valid prefix.

In [26]:
def word_rectangle(words: list[str]) -> list[str] | None:
    by_len: dict[int, list[str]] = defaultdict(list)
    for w in set(words):
        by_len[len(w)].append(w)
    tries: dict[int, dict] = {}
    for L, ws in by_len.items():
        t: dict = {}
        for w in ws:
            node = t
            for c in w:
                node = node.setdefault(c, {})
            node["$"] = True
        tries[L] = t

    def build(h: int, w: int) -> list[str] | None:
        col_trie = tries[h]
        rows: list[str] = []

        def backtrack(col_nodes: list[dict]) -> bool:
            if len(rows) == h:
                return all("$" in node for node in col_nodes)
            for word in by_len[w]:
                nxt = []
                for c, node in zip(word, col_nodes):
                    child = node.get(c)
                    if child is None:
                        break
                    nxt.append(child)
                else:
                    rows.append(word)
                    if backtrack(nxt):
                        return True
                    rows.pop()
            return False

        return rows if backtrack([col_trie] * w) else None

    max_len = max(by_len)
    for area in range(max_len * max_len, 0, -1):
        for h in range(1, max_len + 1):
            if area % h:
                continue
            w = area // h
            if w > max_len or h not in by_len or w not in by_len:
                continue
            if (res := build(h, w)) is not None:
                return res
    return None


vocab = ["ant", "bee", "cat", "nee", "tee", "ace", "abc", "net", "tan", "eat", "cab", "bat",
         "ate", "tea", "ten", "at", "an", "a"]
rect = word_rectangle(vocab)
cols = ["".join(r[i] for r in rect) for i in range(len(rect[0]))]
assert all(r in vocab for r in rect) and all(c in vocab for c in cols)
print("\n".join(rect), f"\n({len(rect)}x{len(rect[0])})")

cat
ace
tee 
(3x3)


## 17.26 Sparse Similarity
Two documents with distinct word IDs have similarity `|A ∩ B| / |A ∪ B|` (Jaccard). Given many documents, print every pair with similarity > 0. The data is sparse, so build an **inverted index** from word to docs and count intersections only for pairs that actually share a word. The union then follows from the sizes: `|A| + |B| - |A∩B|`.

In [27]:
def sparse_similarity(docs: dict[int, list[int]]) -> dict[tuple[int, int], float]:
    index: dict[int, list[int]] = defaultdict(list)
    for doc_id, words in docs.items():
        for w in set(words):
            index[w].append(doc_id)
    intersections: Counter[tuple[int, int]] = Counter()
    for doc_ids in index.values():
        doc_ids.sort()
        for a, b in itertools.combinations(doc_ids, 2):
            intersections[(a, b)] += 1
    sizes = {d: len(set(ws)) for d, ws in docs.items()}
    return {pair: inter / (sizes[pair[0]] + sizes[pair[1]] - inter) for pair, inter in intersections.items()}


docs = {13: [14, 15, 100, 9, 3], 16: [32, 1, 9, 3, 5], 19: [15, 29, 2, 6, 8, 7], 24: [7, 10]}
sims = sparse_similarity(docs)
assert sims == {(13, 16): 0.25, (13, 19): 0.1, (19, 24): 1 / 7}
for (a, b), s in sorted(sims.items()):
    print(f"{a}, {b} : {s:.4f}")

13, 16 : 0.2500
13, 19 : 0.1000
19, 24 : 0.1429
